# Wake Word Experiments
> **Systematic comparison of tiers, loss functions, and augmentation strategies.**  
> Synthesise a dataset once, then train multiple configurations and compare F1 / EER in a results table and plots.

This notebook complements `kaggle_quickstart.ipynb` (single-run) and `genetic_search.ipynb` (HP search). Use it when you want to understand *which combination of architecture, loss, and augmentation works best for your wake word*.

---

## What this notebook does

| Step | Cell | What happens |
|------|------|--------------|
| 1. Config | 2 | Define the experiment matrix |
| 2. Install | 3 | Install dependencies; auto-detect platform |
| 3. MLflow | 4 | Inject Kaggle Secrets; create a shared MLflow experiment |
| 4. Dataset | 5 | Download / synthesise the dataset once (shared across all runs) |
| 5. Grid run | 6 | Loop over (tier × loss × augmentation) — resumable |
| 6. Results table | 7 | Pandas DataFrame of all results |
| 7. Plots | 8 | F1 bar chart + ROC overlay for top-3 configs |
| 8. Best model | 9 | Print best config, verify ONNX files |

---

## Quick start

**Minimum viable run (no config needed):** click **Run All** — trains 4 cells (micro × bce × none, micro × bce × bg_noise, small × bce × none, small × bce × bg_noise) in ~30 min on Kaggle GPU T4.

**Skip a dimension:** set `TIERS_TO_TRAIN=micro` or `LOSSES_TO_TEST=bce` to reduce the grid.

---

## Configuration reference

### Core
| Variable | Default | Purpose |
|----------|---------|---------|
| `WAKE_WORD` | `hey jarvis` | Target phrase |
| `OUTPUT_DIR` | `./ww_output` | All outputs land here |
| `DEVICE` | `auto` | `auto`, `cpu`, or `cuda` |
| `SEED` | `42` | Random seed |

### Dataset
| Variable | Default | Purpose |
|----------|---------|---------|
| `N_POSITIVE` | `500` | TTS samples to synthesise |
| `LANG` | `en` | BCP-47 language for TTS |
| `ADVERSARIAL` | `true` | Phonetically-similar hard negatives |
| `DOWNLOAD_AUGMENT` | `true` | Download bg-noise / music / RIR from HF |
| `CUSTOM_TRAIN_CSV` | *(empty)* | BYO dataset — skips TTS synthesis |
| `CUSTOM_TEST_CSV` | *(empty)* | BYO test split (auto 80/20 if absent) |

### Experiment grid
| Variable | Default | Purpose |
|----------|---------|---------|
| `TIERS_TO_TRAIN` | `micro,small` | Comma-separated tier list |
| `LOSSES_TO_TEST` | `bce,focal` | Loss functions to compare |
| `AUGMENT_LEVELS` | `none,bg_noise` | Augmentation conditions: `none`, `bg_noise`, `full` |
| `EPOCHS_PER_RUN` | `30` | Epochs per experiment cell |
| `BATCH_SIZE` | `32` | Training batch size |
| `SKIP_COMPLETED` | `true` | Skip cells with saved results (resumable) |

### MLflow
| Variable | Default | Purpose |
|----------|---------|---------|
| `MLFLOW_URI` | *(empty)* | MLflow tracking URI (optional) |
| `MLFLOW_SECRET` | `MLFLOW_TOKEN` | Kaggle Secret name for MLflow token |
| `MLFLOW_EXPERIMENT` | `ww_experiments` | MLflow experiment name |

---

## Tier reference

| Tier | Extractor | Params | Target hardware |
|------|-----------|--------|-----------------|
| `micro` | MFCC-20 | ~50 K | MCU / RPi Zero |
| `small` | MFCC-40 | ~200 K | RPi 3/4 |
| `filterbank_small` | FilterBank | ~250 K | Embedded SBC |
| `gammatone_small` | Gammatone | ~250 K | Embedded SBC |
| `sincnet_small` | SincNet | ~350 K | Low-power CPU |
| `delta_micro` | MFCC+Δ+ΔΔ | ~75 K | MCU with more flash |

## Loss function reference

| Loss | Notes |
|------|-------|
| `bce` | Binary cross-entropy — fast baseline |
| `focal` | Focal loss — handles class imbalance better than BCE |
| `rppl` | Robust Prototype Diversity Loss — best with large NWW pools |
| `arcface` | ArcFace margin — metric-learning style |
| `supcon` | Supervised contrastive — requires good positives |

## Augmentation levels

| Level | What is applied |
|-------|-----------------|
| `none` | No augmentation |
| `bg_noise` | Background noise only (requires `DOWNLOAD_AUGMENT=true` or `BG_NOISE_DIR`) |
| `full` | Background noise + music + room impulse response (all three) |

---

## Kaggle free-tier budget

Approximate wall-clock time on GPU T4 (30 epochs, `small` tier):
- ~8 min/cell with bg_noise augmentation
- Default grid (2 tiers × 2 losses × 2 augment levels = 8 cells) ≈ **60–80 min**
- Full grid (6 tiers × 5 losses × 3 augment levels = 90 cells) ≈ **12 h** — well within 30 h/week quota if spread across sessions

Use `SKIP_COMPLETED=true` (default) to pause and resume safely across Kaggle sessions.

## Cell 2 — Configuration
Edit this cell or set environment variables / Kaggle Secrets.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "500"))
LANG             = os.environ.get("LANG",             "en")
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
CUSTOM_TRAIN_CSV = os.environ.get("CUSTOM_TRAIN_CSV", "")
CUSTOM_TEST_CSV  = os.environ.get("CUSTOM_TEST_CSV",  "")

# ── Experiment grid ───────────────────────────────────────────────────────────
TIERS_TO_TRAIN   = os.environ.get("TIERS_TO_TRAIN",   "micro,small").split(",")
LOSSES_TO_TEST   = os.environ.get("LOSSES_TO_TEST",   "bce,focal").split(",")
AUGMENT_LEVELS   = os.environ.get("AUGMENT_LEVELS",   "none,bg_noise").split(",")
EPOCHS_PER_RUN   = int(os.environ.get("EPOCHS_PER_RUN",  "30"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "32"))
SKIP_COMPLETED   = os.environ.get("SKIP_COMPLETED",   "true").lower() == "true"

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "ww_experiments")

# ── Derived ───────────────────────────────────────────────────────────────────
N_CELLS = len(TIERS_TO_TRAIN) * len(LOSSES_TO_TEST) * len(AUGMENT_LEVELS)
print(f"Experiment grid: {len(TIERS_TO_TRAIN)} tiers × {len(LOSSES_TO_TEST)} losses "
      f"× {len(AUGMENT_LEVELS)} augment levels = {N_CELLS} cells")
print(f"Tiers  : {TIERS_TO_TRAIN}")
print(f"Losses : {LOSSES_TO_TEST}")
print(f"Augment: {AUGMENT_LEVELS}")

## Cell 3 — Install & platform detection

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

## Cell 4 — MLflow setup

In [ ]:
import os

if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"No Kaggle secret '{MLFLOW_SECRET}' found — MLflow disabled. ({e})")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    # Create experiment if it doesn't exist
    try:
        import mlflow
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        print(f"MLflow experiment: {MLFLOW_EXPERIMENT!r} at {MLFLOW_URI}")
    except Exception as e:
        print(f"MLflow setup warning: {e}")
else:
    print("MLFLOW_URI not set — experiment tracking disabled")

## Cell 5 — Dataset

The dataset is synthesised **once** and shared across all experiment cells. Set `CUSTOM_TRAIN_CSV` to skip synthesis entirely.

In [ ]:
import shutil
from pathlib import Path

# Disk space guard — experiments need more room than a single run
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 5, f"Only {free_gb:.1f} GB free — need at least 5 GB for experiment outputs."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}  # populated below for 'bg_noise' and 'full' augment levels

if CUSTOM_TRAIN_CSV:
    from ww_trainer.utils import read_dataset_csv
    import random, csv as _csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    if CUSTOM_TEST_CSV:
        test_csv = Path(CUSTOM_TEST_CSV)
    else:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED)
            random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for path, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(path, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
    print(f"BYO mode: train={train_csv}, test={test_csv}")
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word

    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"

    if _train_csv_check.exists():
        print(f"Reusing existing dataset at {dataset_dir}")
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
    else:
        print(f"Running datagen pipeline for '{WAKE_WORD}'...")
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD,
            output_dir=dataset_dir,
            n_positive=N_POSITIVE,
            lang=LANG,
            adversarial=ADVERSARIAL,
            vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT,
            seed=SEED,
        ))

    train_csv = _dr.train_csv
    test_csv  = _dr.test_csv

    # Cache augmentation paths for use in the experiment grid
    if hasattr(_dr, "bg_noise_dir") and _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
        _aug_kwargs_full["bg_noise_folder"] = str(_dr.bg_noise_dir)
    if hasattr(_dr, "music_dir") and _dr.music_dir and Path(_dr.music_dir).exists():
        _aug_kwargs_full["music_folder"] = str(_dr.music_dir)
    if hasattr(_dr, "rir_dir") and _dr.rir_dir and Path(_dr.rir_dir).exists():
        _aug_kwargs_full["rir_folder"] = str(_dr.rir_dir)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")
print(f"Available augmentation dirs: {list(_aug_kwargs_full.keys())}")

## Cell 6 — Experiment grid

Loops over all (tier × loss × augmentation) combinations. Each cell:
1. Checks if a result already exists (and skips if `SKIP_COMPLETED=true`)
2. Calls `train_from_wakeword()` with the appropriate config
3. Saves the result to `results/`

The loop is **safe to interrupt and resume** — completed cells are never re-run.

In [ ]:
import json
import time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

results_dir = Path(OUTPUT_DIR) / "results"
results_dir.mkdir(parents=True, exist_ok=True)

all_results = []
total = len(TIERS_TO_TRAIN) * len(LOSSES_TO_TEST) * len(AUGMENT_LEVELS)
cell_idx = 0

def _aug_kwargs_for_level(level):
    """Return trainer augmentation kwargs for the requested level."""
    if level == "none":
        return {}
    if level == "bg_noise":
        return {k: v for k, v in _aug_kwargs_full.items() if "bg_noise" in k}
    # full: all three
    return dict(_aug_kwargs_full)


for tier in TIERS_TO_TRAIN:
    for loss in LOSSES_TO_TEST:
        for augment in AUGMENT_LEVELS:
            cell_idx += 1
            cell_key = f"{tier}__{loss}__{augment}"
            result_file = results_dir / f"{cell_key}.json"
            model_subdir = Path(OUTPUT_DIR) / "models" / cell_key

            print(f"\n[{cell_idx}/{total}] tier={tier!r}  loss={loss!r}  augment={augment!r}")

            if SKIP_COMPLETED and result_file.exists():
                saved = json.loads(result_file.read_text())
                print(f"  SKIP (already done): F1={saved.get('f1', '?'):.4f}")
                all_results.append(saved)
                continue

            aug_kw = _aug_kwargs_for_level(augment)
            t0 = time.time()

            try:
                r = train_from_wakeword(
                    WAKE_WORD,
                    str(model_subdir),
                    tier=tier,
                    epochs=EPOCHS_PER_RUN,
                    batch_size=BATCH_SIZE,
                    device=DEVICE,
                    seed=SEED,
                    reuse_dataset=True,
                    losses_cfg=[{"name": loss, "weight": 1.0}],
                    **aug_kw,
                )
                elapsed = time.time() - t0
                row = {
                    "tier": tier,
                    "loss": loss,
                    "augment": augment,
                    "f1": r.metrics.get("f1", 0.0),
                    "precision": r.metrics.get("precision", 0.0),
                    "recall": r.metrics.get("recall", 0.0),
                    "elapsed_s": round(elapsed, 1),
                    "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
                    "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
                    "status": "ok",
                }
                print(f"  DONE: F1={row['f1']:.4f}  P={row['precision']:.4f}  R={row['recall']:.4f}  ({elapsed:.0f}s)")
            except Exception as exc:
                elapsed = time.time() - t0
                row = {
                    "tier": tier, "loss": loss, "augment": augment,
                    "f1": 0.0, "precision": 0.0, "recall": 0.0,
                    "elapsed_s": round(elapsed, 1),
                    "head_onnx": "", "feat_onnx": "",
                    "status": f"error: {exc}",
                }
                print(f"  ERROR: {exc}")

            result_file.write_text(json.dumps(row, indent=2))
            all_results.append(row)

print(f"\nGrid complete: {len(all_results)} cells ({sum(1 for r in all_results if r['status']=='ok')} succeeded)")

## Cell 7 — Results table

In [ ]:
import pandas as pd
from pathlib import Path
import json

# Reload all results from disk (handles cases where the loop was partially run earlier)
_loaded = []
for f in sorted((Path(OUTPUT_DIR) / "results").glob("*.json")):
    _loaded.append(json.loads(f.read_text()))

df = pd.DataFrame(_loaded)
df = df.sort_values("f1", ascending=False).reset_index(drop=True)

# Display columns
_display_cols = ["tier", "loss", "augment", "f1", "precision", "recall", "elapsed_s", "status"]
_display_cols = [c for c in _display_cols if c in df.columns]

print(f"Results ({len(df)} cells):")
display(df[_display_cols].style.format({"f1": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}"}))

best = df[df["status"] == "ok"].iloc[0] if (df["status"] == "ok").any() else None
if best is not None:
    print(f"\nBest config: tier={best['tier']!r}, loss={best['loss']!r}, augment={best['augment']!r}")
    print(f"  F1={best['f1']:.4f}  P={best['precision']:.4f}  R={best['recall']:.4f}")

## Cell 8 — Plots

- **F1 bar chart**: all configs sorted by F1, colour-coded by augmentation level
- **Precision–recall scatter**: each config as a point
- **Top-3 config summary**: table with F1, P, R, elapsed

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd

df_ok = df[df["status"] == "ok"].copy()

if df_ok.empty:
    print("No successful runs to plot yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Wake word: {WAKE_WORD!r}", fontsize=13)

    # ── F1 bar chart ──────────────────────────────────────────────────────────
    ax = axes[0]
    labels = [f"{r.tier}\n{r.loss}\n{r.augment}" for _, r in df_ok.iterrows()]
    aug_levels = sorted(df_ok["augment"].unique())
    aug_colors = {a: cm.tab10(i) for i, a in enumerate(aug_levels)}
    colors = [aug_colors[a] for a in df_ok["augment"]]
    bars = ax.bar(range(len(df_ok)), df_ok["f1"], color=colors, edgecolor="white", linewidth=0.5)
    ax.set_xticks(range(len(df_ok)))
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylabel("F1")
    ax.set_title("F1 by configuration")
    ax.set_ylim(0, 1.05)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=aug_colors[a], label=a) for a in aug_levels],
              title="Augment", fontsize=8)
    for i, (bar, f1) in enumerate(zip(bars, df_ok["f1"])):
        ax.text(bar.get_x() + bar.get_width()/2, f1 + 0.01, f"{f1:.3f}",
                ha="center", va="bottom", fontsize=7)

    # ── Precision-Recall scatter ───────────────────────────────────────────────
    ax2 = axes[1]
    tier_list = sorted(df_ok["tier"].unique())
    tier_markers = {t: m for t, m in zip(tier_list, ["o","s","^","D","v","P"])}
    for _, row in df_ok.iterrows():
        ax2.scatter(row["recall"], row["precision"],
                    marker=tier_markers.get(row["tier"], "o"),
                    color=aug_colors.get(row["augment"], "gray"),
                    s=80, edgecolors="white", linewidth=0.5)
        ax2.annotate(f"{row['loss']}", (row["recall"], row["precision"]),
                     textcoords="offset points", xytext=(4, 3), fontsize=7)
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title("Precision vs Recall")
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_ylim(-0.05, 1.05)
    from matplotlib.lines import Line2D
    ax2.legend(handles=[Line2D([0],[0], marker=tier_markers.get(t,"o"), color="gray",
                               linestyle="None", label=t) for t in tier_list],
               title="Tier", fontsize=8)

    plt.tight_layout()
    plot_path = str(Path(OUTPUT_DIR) / "experiment_results.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot_path}")

    # ── Top-3 summary ─────────────────────────────────────────────────────────
    print("\nTop 3 configurations:")
    print(df_ok[["tier","loss","augment","f1","precision","recall","elapsed_s"]].head(3).to_string(index=False))

## Cell 9 — Best model

Verifies ONNX files for the best configuration and prints copy-paste inference commands.

In [ ]:
from pathlib import Path

if best is None:
    print("No successful runs yet.")
else:
    head_onnx = Path(best["head_onnx"]) if best.get("head_onnx") else None
    feat_onnx = Path(best["feat_onnx"]) if best.get("feat_onnx") else None

    print("Best configuration:")
    print(f"  tier    : {best['tier']}")
    print(f"  loss    : {best['loss']}")
    print(f"  augment : {best['augment']}")
    print(f"  F1      : {best['f1']:.4f}")
    print()
    print("ONNX files:")
    for label, path in [("featurizer", feat_onnx), ("classifier", head_onnx)]:
        if path and path.exists():
            print(f"  OK  {label}: {path} ({path.stat().st_size/1024:.0f} KB)")
        else:
            print(f"  MISSING  {label}: {path}")

    print()
    print("Quick inference (Python):")
    print("  from ww_trainer.inference import OnnxWakeWordInferencer")
    print(f"  model = OnnxWakeWordInferencer({str(feat_onnx)!r}, {str(head_onnx)!r})")
    print("  score = model.infer(wav_float32_array)")
    print()
    print("Test on a WAV file (CLI):")
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {feat_onnx} \\")
    print(f"      --model      {head_onnx} \\")
    print(f"      --audio      sample.wav")
    print()
    print("Live multi-model dashboard (CLI):")
    models_dir = Path(OUTPUT_DIR) / "models"
    print(f"  .venv/bin/python scripts/eval/listen_all.py --models-dir {models_dir} --max-models 5")